In [8]:
import itertools
import logging
import warnings
import pandas as pd
import numpy as np
from prophet import Prophet
from joblib import Parallel, delayed
from tqdm import tqdm
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Suppress Prophet logs
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)
warnings.filterwarnings('ignore')

# 1. DEFINE EXPANDED HYPERPARAMETER GRID
PARAM_GRID = {
    'changepoint_prior_scale': [0.01, 0.05, 0.1, 0.3, 0.4, 0.5, 0.6],
    'changepoint_range': [0.85, 0.9]
}

GRID_COMBINATIONS = [
    dict(zip(PARAM_GRID.keys(), v)) 
    for v in itertools.product(*PARAM_GRID.values())
]

def tune_single_model(area, category, df_subset):
    """Grid searches parameters using MAE for selection, and logs RMSE."""
    try:
        prophet_df = pd.DataFrame({
            'ds': pd.to_datetime(df_subset['Year'].astype(str) + '-01-01'),
            'y': df_subset['Temperature_Change'].values
        })

        # 90/10 Split Validation
        train_df = prophet_df[prophet_df['ds'].dt.year <= 2013]
        val_df = prophet_df[prophet_df['ds'].dt.year > 2013]

        if len(train_df) < 4 or val_df.empty:
            return None

        best_mae = float('inf')
        best_rmse = float('inf')
        best_params = {'changepoint_prior_scale': 0.05, 'changepoint_range': 0.85}

        for params in GRID_COMBINATIONS:
            try:
                m_temp = Prophet(
                    yearly_seasonality=False,
                    weekly_seasonality=False,
                    daily_seasonality=False,
                    changepoint_prior_scale=params['changepoint_prior_scale'],
                    changepoint_range=params['changepoint_range']
                )
                m_temp.fit(train_df)
                
                preds = m_temp.predict(val_df[['ds']])
                mae = mean_absolute_error(val_df['y'], preds['yhat'])
                rmse = np.sqrt(mean_squared_error(val_df['y'], preds['yhat']))
                
                # Model selection is driven purely by minimizing MAE
                if mae < best_mae:
                    best_mae = mae
                    best_rmse = rmse
                    best_params = params
            except Exception:
                continue

        return {
            'Area': area,
            'Category': category,
            'Validation_MAE': best_mae,
            'Validation_RMSE': best_rmse,
            'Best_CPS': best_params['changepoint_prior_scale'],
            'Best_CPR': best_params['changepoint_range']
        }
    except Exception:
        return None

if __name__ == "__main__":
    print("Loading data for Grid Search...")
    df = pd.read_csv("Cleaned_Environment_Temperature_Data.csv")
    
    season_mapping = {
        'Dec\x96Jan\x96Feb': 'Winter', 'Mar\x96Apr\x96May': 'Spring',
        'Jun\x96Jul\x96Aug': 'Summer', 'Sep\x96Oct\x96Nov': 'Autumn'
    }
    df['Months'] = df['Months'].replace(season_mapping)

    unique_categories = df['Months'].unique()
    all_areas = df['Area'].unique()

    # Filter clean complete areas (59 years)
    clean_areas = [
        area for area in all_areas
        if all(len(df[(df['Area'] == area) & (df['Months'] == cat)]['Year'].unique()) == 59 
               for cat in unique_categories)
    ]

    tasks = [
        (area, cat, df[(df['Area'] == area) & (df['Months'] == cat)])
        for area in clean_areas for cat in unique_categories
    ]

    print(f"Starting Grid Search across {len(tasks)} isolated models (14 param combinations each)...")
    
    results = Parallel(n_jobs=-1)(
        delayed(tune_single_model)(area, cat, data)
        for area, cat, data in tqdm(tasks, desc="Grid Search Auto-Tuning")
    )

    valid_results = [r for r in results if r is not None]
    best_params_df = pd.DataFrame(valid_results)
    
    # Save optimal parameters registry with MAE and RMSE only
    best_params_df.to_csv("best_params_registry.csv", index=False)
    
    print("\n=== GRID SEARCH COMPLETE ===")
    print("Saved optimal parameters to 'best_params_registry.csv'.")
    print(f"Global Average Validation MAE:   {best_params_df['Validation_MAE'].mean():.4f} °C")
    print(f"Global Average Validation RMSE:  {best_params_df['Validation_RMSE'].mean():.4f} °C")

Loading data for Grid Search...
Starting Grid Search across 3162 isolated models (14 param combinations each)...


Grid Search Auto-Tuning: 100%|██████████| 3162/3162 [26:13<00:00,  2.01it/s]



=== GRID SEARCH COMPLETE ===
Saved optimal parameters to 'best_params_registry.csv'.
Global Average Validation MAE:   0.5816 °C
Global Average Validation RMSE:  0.7016 °C


In [9]:
import pandas as pd
df_params = pd.read_csv("best_params_registry.csv")
global_avg_rmse = df_params['Validation_RMSE'].mean()
print(f"Global Average Validation RMSE: {global_avg_rmse:.4f} °C")

Global Average Validation RMSE: 0.7016 °C
